# Environment Setting Up

In [2]:
import os
from dotenv import load_dotenv

# Loading environment variables from .env
load_dotenv()

# Changing directory to main directory for easy data access
working_directory = os.getenv("WORKING_DIRECTORY")
os.chdir(working_directory)

# Checking the change
%pwd

'D:\\Projects\\Coding Agent\\AI-Coding-Research-Agent'

In [3]:
from pathlib import Path

# Checking the change
print("Git folder exists:", Path(".git").exists())

Git folder exists: True


# Agent (Basic)

In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

import asyncio
import os

# Loading environment variables from .env
load_dotenv()

print("Complete")

Complete


In [5]:
os.getenv("OLLAMA_MODEL")

'qwen2.5:14b'

In [6]:
llm = ChatOllama(
    model=os.getenv("OLLAMA_MODEL"),
    temperature="0"
    )

llm.invoke("What is a LLM?")

AIMessage(content='LLM stands for "Master of Laws" or sometimes in the context of technology, it can refer to "Large Language Model." In an academic setting, LLM is a postgraduate law degree that provides specialized education and training in various areas of law. However, given your question seems to be related to recent advancements in AI, you might be referring to "Large Language Model," which is a type of artificial intelligence model designed to process and generate human-like text based on the data it has been trained on. These models are used for tasks such as language translation, text summarization, content creation, and more. If you meant something else by LLM, please provide additional context!', additional_kwargs={}, response_metadata={'model': 'qwen2.5:14b', 'created_at': '2025-11-24T18:58:40.7265936Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6746417400, 'load_duration': 3153967300, 'prompt_eval_count': 35, 'prompt_eval_duration': 46638300, 'eval_count': 138

In [7]:
prompt = "Explain large language models in a simple way."
response = llm.invoke(prompt)

text_output = response.content if hasattr(response, "content") else str(response)
text_output

'Sure! Large Language Models (LLMs) like the one I am based on are advanced computer programs designed to understand and generate human-like text. Imagine you have a very smart friend who can read thousands of books, articles, and websites. This friend has learned so much that they can answer questions, write stories, or even chat with you about almost any topic.\n\nLLMs work in a similar way but are based on complex mathematical models trained on vast amounts of text data from the internet and other sources. They learn patterns in language by analyzing how words and sentences are used together across different contexts. Once they\'ve learned enough, these models can generate new text that sounds natural to humans or answer questions accurately.\n\nThe "large" part refers to the fact that these models have billions of parameters (like settings or rules) which allow them to capture a wide range of language nuances and complexities. This makes them very powerful but also requires signifi

In [8]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from pathlib import Path
from typing import Union
from datetime import datetime

from CodeResearcher.utils.common import create_directories
from CodeResearcher.utils.logger import get_logger

# Initialize application-wide logger
logger = get_logger()


def save_chat_to_pdf(text_output: str, output_pdf_path: Union[Path, str]) -> None:
    """
    Save plain text content into a PDF document.

    Parameters
    text_output str: The text content to write into the PDF.
    
    output_pdf_path Union[Path, str]: The destination path where the PDF should be saved.

    Notes
    - Automatically creates parent directories.
    - Converts Windows Path objects to strings for ReportLab compatibility.
    - Handles multi-line text safely by splitting into Paragraphs.
    """
    # Ensure directory exists, else create it
    create_directories([output_pdf_path.parent], verbose=False)

    try:
        # Create PDF document
        doc = SimpleDocTemplate(str(output_pdf_path), pagesize=letter)
        styles = getSampleStyleSheet()

        # Split text into paragraphs
        story = []
        for line in text_output.split("\n"):
            clean_line = line.strip()
            if clean_line:
                story.append(Paragraph(clean_line, styles["Normal"]))

        if not story:
            raise ValueError("No valid text content to write to PDF.")

        # Write PDF to disk
        doc.build(story)

        logger.info(f"PDF successfully saved: {output_pdf_path}")
        
    except Exception as e:
        logger.error(f"Unable to save PDF file: {e}")


# Example Usage
# Create timestamp for the filename (UTC-friendly)
utc_timestamp = datetime.now().strftime(f"%Y-%m-%d_%H-%M-%S")

# Directory where reports will be stored
save_dir = Path(working_directory) / "research" / "reports"

# Generate a descriptive file name
save_path = save_dir / (f"ollama_response - {utc_timestamp}.pdf")

# Save the PDF
save_chat_to_pdf(text_output, save_path)

[2025-11-25 00:28:46,611: INFO: 3865880445: PDF successfully saved: D:\Projects\Coding Agent\AI-Coding-Research-Agent\research\reports\ollama_response - 2025-11-25_00-28-46.pdf]


In [9]:
# Create a set of parameters for starting an MCP server using standard input/output
server_params = StdioServerParameters(
    # Create executable using - `npx`, or the Node package runner included with Node.js.
    command="npx",

    # Environment variables passed to the launched process.
    env={"FIRECRAWL_API_KEY": os.getenv("FIRECRAWL_API_KEY")},

    # CLI passed to `npx`, which tells it ot run the `firecrawl-mcp` package as an MCP server
    args=["firecrawl-mcp"]
)

In [10]:
async def main():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await load_mcp_tools(session)
            agent = create_react_agent(llm, tools)

            messages = [{
                "role": "system",
                "content": "You are a helpful assistant that can scrape websites, crawl pages, and extract data using Firecrawl tools. Think step by step and use the appropriate tools to help the user.",
            }]

            # Print available tools to the user
            print("Available Tools -", *[tool.name for tool in tools])
            print("-" * 60)

            while True:
                user_input = input("\nUser: ")
                if user_input == "quit":
                    print("Thank you using the Coding Research Agent. Goodbye!")
                    break

                messages.append({"role": "user", "content": user_input[:100_000]})

                try:
                    # Passing to the agent all the messages
                    agent_response = await agent.ainvoke({"messages": messages})

                    # Getting the most recent message from the agent
                    ai_message = agent_response["messages"][-1].content

                    # Printing the agent message to the CLI user
                    print("\nAgent: ", ai_message)

                except Exception as e:
                    print("Error:", e)

In [11]:
await main()

UnsupportedOperation: fileno